Import Necessary Libraries

In [16]:
%matplotlib qt
#The version of the packages used in this notebook are listed in the comments
import numpy as np #1.21.5
import matplotlib.pyplot as plt #3.5.2
import hyperspy.api as hs #1.7.3
from PyQt5.QtWidgets import QFileDialog  #5.12.3
import cv2 #4.5.5
import sys

Define Functions

In [17]:
def noframeimshow(im,contrast=None, greyscale=True, size=5):
    """To display an image without the frame, will pause the program until the figure is closed
    im: (n,m) image to display
    contrast: [min,max] minimum and maximum value to display 
    greyscale: True if the image is greyscale, otherwise use matplotlib default colormap
    size: size of the figure in inches
    """
    if contrast is None:                     #if no contrast is given, use the min and max of the image
        contrast=[np.min(im),np.max(im)]
    y, x = np.shape(im)                      #get the size of the image
    fig = plt.figure(figsize=(size,size*y/x))       #create a figure with the right size
    fig.set_size_inches(size,size*y/x, forward=False)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.axis(False)
    fig.add_axes(ax)
    if greyscale:
        ax.imshow(im,vmin=contrast[0], vmax=contrast[1], cmap='gray')
    else:
        ax.imshow(im,vmin=contrast[0], vmax=contrast[1])
    plt.show()
    fnum=fig.number
    while True:       #wait until the figure is closed        
        plt.pause(0.1)
        if plt.fignum_exists(fnum):
            continue
        else:
            break
    pass


def logimage(im):
    """Logarithmic transform an image
    im: (n,m) image to transform

    return
    im: transformed image
    """
    im=im-np.min(im)   #First linear transform the image to [1,10]
    im=9*im/np.max(im)+1   
    im=np.log10(im)   #Take the log10 of the image, will be in [0,1]
    return im

def update_progress(progress):
    """Print a progress bar"""
    barLength = 10 # Modify this Xto change the length of the progress bar
    status = ""
    if isinstance(progress, int):
        progress = float(progress)
    if not isinstance(progress, float):
        progress = 0
        status = "error: progress var must be float\r\n"
    if progress < 0:
        progress = 0
        status = "Halt...\r\n"
    if progress >= 1:
        progress = 1
        status = "Done...\r\n"
    block = int(round(barLength*progress))
    text = "\rPercent: [{0}] {1}% {2}".format( "#"*block + "-"*(barLength-block), progress*100, status)
    sys.stdout.write(text)
    sys.stdout.flush()

def rebin(arr, new_shape):
    """Rebin an array to a new shape"""
    shape = (new_shape[0], arr.shape[0] // new_shape[0],
             new_shape[1], arr.shape[1] // new_shape[1])
    return arr.reshape(shape).mean(-1).mean(1)

def crop_center(img,cropx,cropy):
    """Crop an image to a new size centered on the original image
    img: (n,m) image to crop
    cropx: new x size
    cropy: new y size"""
    y,x = img.shape
    startx = x//2 - cropx//2
    starty = y//2 - cropy//2    
    return img[starty:starty+cropy, startx:startx+cropx]
 


def dp_median_filtering(data, kernel_size=3):
    """Median filter every diffraction pattern in a 4D dataset
    data: (n,m,kn,km) dataset to filter, in datatype uint16
    kernel_size: size of the median filter kernel"""
    n, m = data.shape[0:2] # Get the dimensions of the dataset
    for i in range(n):
        for j in range(m):
            data[i,j]=cv2.medianBlur(data[i,j], kernel_size).astype(np.uint16) #Apply the median filter
        if i%10 ==0:
            update_progress((i+1)/len(data))
    return data

def local_bkg_removal(data, inner_cutoff=1, outer_cutoff=3, return_locbkg=False):
    """Remove the local background of every diffraction pattern in a 4D dataset
    data: (n,m,kn,km) dataset to remove the background from, in datatype uint16, all values must be greater or equal to 1
    inner_cutoff: (int) inner cutoff length in pixels for the region to calculate the local background
    outer_cutoff: (int) outer cutoff length in pixels for the region to calculate the local background
    return_locbkg: (bool) if True, the local background will be returned
    
    Return
    I_diff: (n,m,kn,km) dataset with the local background removed, in datatype uint16
    loc_bkg: (n,m,kn,km) dataset with the local background, in datatype uint16"""
    data = np.pad(data, ((outer_cutoff, outer_cutoff), (outer_cutoff, outer_cutoff), (0, 0), (0, 0)), mode='reflect') #Pad the dataset by reflecting the edges to allow for the local background calculation at the edges
    n, m, kn, km = data.shape
    I_diff = np.zeros((n-6, m-6, kn, km)) #Initialize the array to store the local background
    if return_locbkg:
        loc_bkg = np.zeros((n-6, m-6, kn, km))
    for i in range(len(I_diff)):
        for j in range(len(I_diff[0])):
            d_outer = 2*outer_cutoff+1 #Calculate the size of the region to calculate the local background
            d_inner = 2*inner_cutoff+1 #Calculate the size of the region to left out of the local background calculation
            locbkg = (np.sum(data[i:i+d_outer, j:j+d_outer], axis=(0,1))-np.sum(data[i+outer_cutoff-inner_cutoff:i+outer_cutoff-inner_cutoff+d_inner, j+outer_cutoff-inner_cutoff:j+outer_cutoff-inner_cutoff+d_inner], axis=(0,1)))/(d_outer**2-d_inner**2) #Calculate the local background
            if return_locbkg:
                loc_bkg[i,j] = locbkg
            try:
                I_diff[i,j] = np.log(data[i+outer_cutoff, j+outer_cutoff])-np.log(locbkg) # Logarithmic subtraction of the local background
            except RuntimeWarning:
                print('RuntimeWarning')
                break
            # loglocbkg[i,j] = logimage(bindata[i+2, j+2])-logimage(locbkg)
            # _ = logdp(locbkg)
        update_progress(i/(n-5))
    I_diff = uint16_transform(I_diff) #Transform the dataset to uint16
    if return_locbkg:
        loc_bkg = uint16_transform(loc_bkg)
        return I_diff, loc_bkg
    else:
        return I_diff

def uint16_transform(data):
    """Transform a dataset to uint16 datatype
    data: (n,m,kn,km) dataset to transform

    return
    data: transformed dataset, will have a value between 1 and 65535 for futrue logarithmic tranformations or subtractions"""
    data = data-np.min(data) #Linearly shift the dataset so the minimum value is 0
    data = 65534*data/np.max(data) #Linearly scale the dataset so the maximum value is 65534
    data = data+1 #Add 1 to the dataset so the value is in [1,65535]
    data = data.astype(np.uint16) #Transform the dataset to uint16
    print("Transformed to uint16 datatype")
    return data



Load the raw data (Using .emi file as example)

In [18]:
#Create a qt window to get the selected file name
fileName = QFileDialog.getOpenFileName( caption="Open Image", filter="Raw Images (*.tif) ;; Image Files (*.png *.jpg *.bmp) ;; EMI Files (*.emi) ;; Numpy Files (*.npy)", initialFilter="EMI Files (*.emi)")[0]
#Remove the extension from the file name to use it later
filename = fileName[:-4]

In [19]:
"""Here we need to load the dataset to the array 'data' with dimension (n,m,kn,km). The example uses the hypserspy packge to load an .emi file. User can substitute this part with their own code to load the dataset as long as the array 'data' is created properly."""

s = hs.load(fileName) #load the file using hyperspy
data = np.array(s.data) #get the data from the hyperspy object to a numpy array
del s #delete the hyperspy object to free up memory



'The noframimshow function will pause the program until the figure is closed. This is useful to check the pattern before continuing with the more time consuming steps.\nIf one wish to not pause the program, one can use the matplotlib function plt.imshow() instead of noframeimshow()'

In [ ]:
N, M , kN, kM = np.shape(data) #get the dimensions of the dataset
dpmax_raw = np.max(data, axis=(0,1)) #Get the maximum value of each pixel in the diffraction pattern stack. Useful for initial visualization of the all the diffraction peaks and hot pixels and determining the size of the diffraction pattern we want to crop out

"""The noframimshow function will pause the program until the figure is closed. This is useful to check the pattern before continuing with the more time consuming steps.
If one wish to not pause the program, one can use the matplotlib function plt.imshow() instead of noframeimshow()"""

noframeimshow(logimage(dpmax_raw)) #Display the maximum value of each pixel in the diffraction pattern stack


Binning and cropping the diffraction patterns to reduce file size

In [20]:
#Determine whether the diffraction pattern needs to be binned and cropped
binned = False # The example dataset is already binned on the detector end to desired size (256x256), so we set binned to False
cropped = True # The example dataset can be cropped to (150x150), so we set cropped to True
bin_size = 256 #The size of the diffraction pattern after binning, assuming the diffraction pattern is square
crop_size = 150 #The size of the diffraction pattern after cropping, assuming the diffraction pattern is square
#If both needed, we will do binning first, then cropping. So the crop_size will be the final size of the processed diffraction pattern

if binned:
    if cropped:
        bindata = np.zeros((N,M,crop_size,crop_size)) #Initialize the array to store the binned and cropped diffraction patterns
        for i in range(N):
            for j in range(M):
                bindata[i,j] = crop_center(rebin(data[i,j,:,:],(bin_size,bin_size)), crop_size, crop_size) #Rebin the diffraction pattern, then crop it
            update_progress(i/(N-1))
    else:
        bindata = np.zeros((N,M,bin_size,bin_size))
        for i in range(N):
            for j in range(M):
                bindata[i,j] = rebin(data[i,j,:,:],(bin_size,bin_size))
            update_progress(i/(N-1))
else:
    if cropped:
        bindata = np.zeros((N,M,crop_size,crop_size))
        for i in range(N):
            for j in range(M):
                bindata[i,j] = crop_center(data[i,j,:,:], crop_size, crop_size)
            update_progress(i/(N-1))
    else:
        bindata = data.copy()

del data #delete the original dataset to free up memory

Percent: [##########] 100% Done...18592%  


Transform the processed data to uint16 datatype to reduce file size

In [21]:
bindata = uint16_transform(bindata) #Transform the dataset to uint16 datatype

Transformed to uint16 datatype


Apply medium filter to each diffraction pattern to remove hot pixels

In [22]:
bindata = dp_median_filtering(bindata, 3) #Apply median filtering to the dataset

dpmax = np.max(bindata, axis=(0,1)) #Get the maximum value of each pixel in the preprocessed diffraction pattern stack
noframeimshow(logimage(dpmax)) #Display the maximum value of each pixel in the preprocessed diffraction pattern stack, check if the binning and cropping is done correctly and if the hot pixels are removed


Percent: [##########] 95.5% 00000000001% 

Remove local background

In [23]:
#I_diff = local_bkg_removal(bindata, 1, 3) #Remove the local background from the dataset
I_diff, loc_bkg = local_bkg_removal(bindata, 1, 3, return_locbkg=True) #Remove the local background from the dataset, also return the local background to check the quality of the background removal

In [28]:
#Plot one of the diffraction patterns in original data, the local background, and the data with the local background removed, to check the quality of the background removal
plt.subplots(1,3)
plt.subplot(1,3,1)
im = logimage(logimage(bindata[10,10,:,:]))
plt.imshow(im)
plt.clim(np.min(im),np.max(im)*0.5)
plt.title('Original')
plt.subplot(1,3,2)
im = logimage(logimage(loc_bkg[10,10,:,:]))
plt.imshow(im)
plt.clim(np.min(im),np.max(im)*0.5)
plt.title('Local Background')
plt.subplot(1,3,3)
plt.imshow(I_diff[10,10,:,:])
plt.title('Local Background Removed')
plt.show()

Save the preprocessed data and the diffuse scattering data (local background removed data) to the same directory as the raw data in numpy (.npy) file format

In [25]:
n, m, kn, km = bindata.shape
filename = filename+f'preprocessed_{n}x{m}x{kn}x{km}'
np.save(filename+'.npy', bindata)
np.save(filename+'_I_diff.npy', I_diff)